# 🤰 Onamiz — AI Model Training (v2)

**Maqsad:** Homiladorlik simptomlarini tahlil qilib, doktorga borish kerakligini aniqlash

**Ma'lumot manbalari:**
- WHO Antenatal Care Guidelines 2016
- ACOG Practice Bulletins 2023
- FIGO Pre-eclampsia Guidelines 2022
- Edinburgh Postnatal Depression Scale (EPDS)
- Kaggle: Maternal Health Risk Dataset (1014 ta namuna)
- Kaggle: Fetal Health CTG Dataset (2126 ta namuna)

**Model chiqaradi:**
- 🟢 `yashil` — Xavfsiz, rejali ko'rikka boring
- 🟡 `sariq` — Diqqat talab, bu hafta shifokorni ko'ring
- 🔴 `qizil` — Yuqori xavf, bugun boring
- 🚨 `favqulodda` — Hoziroq tez yordam

**Muhim:** Model hukm chiqarmaydi, faqat yo'naltiradi.

## 📦 1-QADAM: Kutubxonalarni o'rnatish

In [ ]:
!pip install -q pandas numpy scikit-learn xgboost lightgbm joblib matplotlib seaborn imbalanced-learn shap

## 📂 2-QADAM: Ma'lumotlarni yuklash

Ikkita usul:
- **A)** Google Drive'dan yuklash
- **B)** Kaggle API orqali yuklash

In [ ]:
# === USUL A: Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/onamiz_data'
os.makedirs(DATA_DIR, exist_ok=True)
print('Drive ulandi:', DATA_DIR)

In [ ]:
# === USUL B: Kaggle API (kaggle.json kerak) ===
# from google.colab import files
# files.upload()  # kaggle.json faylini yuklang

# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d csafrit2/maternal-health-risk-data -p /content/data --unzip
# !kaggle datasets download -d andrewmvd/fetal-health-classification -p /content/data --unzip

## 🔬 3-QADAM: Importlar va sozlamalar

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
plt.style.use('seaborn-v0_8-whitegrid')
print('✅ Kutubxonalar tayyor')

## 🏗️ 4-QADAM: Simptom-asosli dataset yaratish

WHO + ACOG + FIGO protokollaridan olingan bilimlar asosida sintetik dataset generatsiya qilamiz.
Bu dataset real klinik bilimlarni aks ettiradi.

In [ ]:
# =============================================================
# WHO/ACOG/FIGO protokollaridan olingan xavf ballari
# Har bir simptom kombinatsiyasi → xavf darajasi
# =============================================================

np.random.seed(RANDOM_STATE)
N = 3000  # namunalar soni

def generate_symptom_dataset(n_samples=3000):
    """WHO/ACOG/FIGO protokollariga asoslangan sintetik dataset."""
    
    records = []
    
    for _ in range(n_samples):
        # Asosiy ma'lumotlar
        trimester = np.random.choice(['T1','T2','T3'], p=[0.35, 0.35, 0.30])
        age = np.random.randint(16, 45)
        gestational_week = {
            'T1': np.random.randint(4, 13),
            'T2': np.random.randint(13, 27),
            'T3': np.random.randint(27, 41)
        }[trimester]
        
        # 1-TRIMEST SIMPTOMLAR (WHO ANC + ACOG)
        vaginal_bleeding = np.random.choice([0,1,2], p=[0.80, 0.15, 0.05])  # 0=yo'q,1=ozgina,2=ko'p
        one_sided_pain   = np.random.choice([0,1],   p=[0.93, 0.07])         # ektopik xavf
        nausea_severity  = np.random.choice([0,1,2,3,4], p=[0.20,0.30,0.30,0.15,0.05])  # 0-4
        urinary_burning  = np.random.choice([0,1],   p=[0.85, 0.15])         # UTI
        fever            = np.random.choice([0,1,2], p=[0.85, 0.10, 0.05])   # 0=normal,1=subfebril,2=yuqori
        dizziness        = np.random.choice([0,1,2], p=[0.70, 0.20, 0.10])   # anemiya
        prev_miscarriage = np.random.choice([0,1,2], p=[0.75, 0.15, 0.10])   # 0=yo'q,1=1x,2=2+
        
        # 2-TRIMEST SIMPTOMLAR (Preeklampsia - FIGO 2022)
        headache_severity = np.random.choice([0,1,2], p=[0.70, 0.20, 0.10])  # 0=yo'q,1=ozgina,2=kuchli
        visual_disturbance = np.random.choice([0,1],  p=[0.95, 0.05])         # eklampsiya
        edema_level      = np.random.choice([0,1,2,3], p=[0.55,0.25,0.15,0.05])  # shish darajasi
        fetal_movement   = np.random.choice([0,1,2], p=[0.80, 0.15, 0.05])   # 0=yaxshi,1=kam,2=yo'q
        epigastric_pain  = np.random.choice([0,1,2], p=[0.80, 0.15, 0.05])   # HELLP sindrom
        sudden_weight_gain = np.random.choice([0,1], p=[0.90, 0.10])          # 3+kg/hafta
        fluid_leaking    = np.random.choice([0,1],   p=[0.97, 0.03])          # PPROM
        
        # 3-TRIMEST SIMPTOMLAR
        fetal_movement_t3 = np.random.choice([0,1,2], p=[0.78, 0.17, 0.05])  # <10 harakat/12soat
        contractions      = np.random.choice([0,1,2], p=[0.75, 0.17, 0.08])  # erta tug'ruq
        bleeding_with_pain = np.random.choice([0,1],  p=[0.97, 0.03])         # plasenta ajralishi
        shortness_of_breath = np.random.choice([0,1,2], p=[0.75, 0.20, 0.05]) # o'pka shishi
        
        # Vital belgilar (o'lchab olinadigan)
        systolic_bp  = np.random.normal(115, 15)
        diastolic_bp = np.random.normal(75, 10)
        heart_rate   = np.random.normal(82, 12)
        
        # ============================================================
        # XAVF HISOBLASH (WHO/ACOG/FIGO protokollariga asoslanib)
        # ============================================================
        risk_score = 0
        
        # --- FAVQULODDA holatlar (darhol) ---
        if vaginal_bleeding == 2:     risk_score += 10  # ko'p qon ketish
        if one_sided_pain == 1:       risk_score += 10  # ektopik ehtimol
        if visual_disturbance == 1:   risk_score += 10  # eklampsiya
        if fetal_movement == 2:       risk_score += 10  # homila harakatsiz
        if fluid_leaking == 1:        risk_score += 10  # PPROM
        if bleeding_with_pain == 1:   risk_score += 10  # plasenta ajralishi
        if fetal_movement_t3 == 2:    risk_score += 10  # gipoksiya
        
        # --- YUQORI XAVF (bugun boring) ---
        if headache_severity == 2:    risk_score += 5
        if epigastric_pain == 2:      risk_score += 5
        if fever == 2:                risk_score += 4
        if nausea_severity == 4:      risk_score += 4
        if contractions == 2:         risk_score += 5
        if shortness_of_breath == 2:  risk_score += 4
        if systolic_bp >= 140:        risk_score += 5  # preeklampsia chegarasi
        if diastolic_bp >= 90:        risk_score += 5
        if systolic_bp >= 160:        risk_score += 5  # og'ir preeklampsia
        
        # --- O'RTA XAVF (bu hafta boring) ---
        if vaginal_bleeding == 1:     risk_score += 3
        if nausea_severity == 3:      risk_score += 3
        if urinary_burning == 1:      risk_score += 3
        if dizziness == 2:            risk_score += 3
        if headache_severity == 1:    risk_score += 2
        if edema_level == 2:          risk_score += 2
        if edema_level == 3:          risk_score += 3
        if fetal_movement == 1:       risk_score += 3
        if fetal_movement_t3 == 1:    risk_score += 3
        if sudden_weight_gain == 1:   risk_score += 2
        if prev_miscarriage == 2:     risk_score += 2
        if fever == 1:                risk_score += 2
        if contractions == 1:         risk_score += 2
        if epigastric_pain == 1:      risk_score += 2
        if shortness_of_breath == 1:  risk_score += 2
        if age < 18 or age > 40:      risk_score += 1
        if prev_miscarriage == 1:     risk_score += 1
        
        # --- XAVF DARAJASINI BELGILASH ---
        if risk_score >= 10:
            risk_level = 'emergency'  # favqulodda
        elif risk_score >= 6:
            risk_level = 'high'       # qizil
        elif risk_score >= 3:
            risk_level = 'medium'     # sariq
        else:
            risk_level = 'low'        # yashil
        
        records.append({
            # Asosiy
            'trimester': trimester,
            'age': round(age),
            'gestational_week': gestational_week,
            # 1-trimest
            'vaginal_bleeding': vaginal_bleeding,
            'one_sided_pain': one_sided_pain,
            'nausea_severity': nausea_severity,
            'urinary_burning': urinary_burning,
            'fever': fever,
            'dizziness': dizziness,
            'prev_miscarriage': prev_miscarriage,
            # 2-trimest
            'headache_severity': headache_severity,
            'visual_disturbance': visual_disturbance,
            'edema_level': edema_level,
            'fetal_movement': fetal_movement,
            'epigastric_pain': epigastric_pain,
            'sudden_weight_gain': sudden_weight_gain,
            'fluid_leaking': fluid_leaking,
            # 3-trimest
            'fetal_movement_t3': fetal_movement_t3,
            'contractions': contractions,
            'bleeding_with_pain': bleeding_with_pain,
            'shortness_of_breath': shortness_of_breath,
            # Vital
            'systolic_bp': round(systolic_bp, 1),
            'diastolic_bp': round(diastolic_bp, 1),
            'heart_rate': round(heart_rate, 1),
            # Target
            'risk_level': risk_level
        })
    
    return pd.DataFrame(records)

df = generate_symptom_dataset(3000)
print('Dataset shakli:', df.shape)
print('\nXavf taqsimoti:')
print(df['risk_level'].value_counts())
df.head()

In [ ]:
# Ma'lumotlarni ko'rish
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Xavf taqsimoti
colors = ['#27ae60','#f39c12','#e74c3c','#8e44ad']
df['risk_level'].value_counts().plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Xavf darajasi taqsimoti', fontsize=13, fontweight='bold')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)

# 2. Trimest bo'yicha
df.groupby(['trimester','risk_level']).size().unstack().plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Trimest bo\'yicha xavf', fontsize=13, fontweight='bold')
axes[1].set_xlabel('')

# 3. Yosh bo'yicha
df.boxplot(column='age', by='risk_level', ax=axes[2])
axes[2].set_title('Yosh vs Xavf darajasi', fontsize=13, fontweight='bold')
axes[2].set_xlabel('')
plt.suptitle('')
plt.tight_layout()
plt.savefig('/content/data_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Grafik saqlandi')

## ⚙️ 5-QADAM: Ma'lumotlarni tayyorlash

In [ ]:
# Kategorik o'zgaruvchilarni raqamlarga aylantirish
le_trimester = LabelEncoder()
df['trimester_enc'] = le_trimester.fit_transform(df['trimester'])

le_target = LabelEncoder()
df['risk_enc'] = le_target.fit_transform(df['risk_level'])

print('Sinflar:', dict(enumerate(le_target.classes_)))

FEATURE_COLS = [
    'trimester_enc', 'age', 'gestational_week',
    'vaginal_bleeding', 'one_sided_pain', 'nausea_severity',
    'urinary_burning', 'fever', 'dizziness', 'prev_miscarriage',
    'headache_severity', 'visual_disturbance', 'edema_level',
    'fetal_movement', 'epigastric_pain', 'sudden_weight_gain', 'fluid_leaking',
    'fetal_movement_t3', 'contractions', 'bleeding_with_pain', 'shortness_of_breath',
    'systolic_bp', 'diastolic_bp', 'heart_rate'
]

X = df[FEATURE_COLS]
y = df['risk_enc']

print(f'Features: {X.shape[1]} ta')
print(f'Namunalar: {X.shape[0]} ta')
print(f'\nSinf taqsimoti:\n{pd.Series(le_target.inverse_transform(y)).value_counts()}')

In [ ]:
# SMOTE — kam uchraydigan sinflarni balanslash (emergency sinfni ko'paytirish)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

smote = SMOTE(random_state=RANDOM_STATE)
X_balanced, y_balanced = smote.fit_resample(X_scaled, y)

print('SMOTE dan keyin:')
print(pd.Series(le_target.inverse_transform(y_balanced)).value_counts())

# Train/test split
X_tr, X_te, y_tr, y_te = train_test_split(
    X_balanced, y_balanced,
    test_size=0.2, stratify=y_balanced, random_state=RANDOM_STATE
)
print(f'\nTrain: {X_tr.shape[0]} | Test: {X_te.shape[0]}')

## 🏆 6-QADAM: Modellarni o'qitish va tanlash

In [ ]:
candidates = {
    'RandomForest': RandomForestClassifier(
        n_estimators=300, max_depth=10,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=6,
        learning_rate=0.1, eval_metric='mlogloss',
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=300, max_depth=6,
        learning_rate=0.1, random_state=RANDOM_STATE,
        n_jobs=-1, verbose=-1
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=5,
        learning_rate=0.1, random_state=RANDOM_STATE
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

print('=' * 60)
print('  MODELLARNI BAHOLASH — 5-fold Cross Validation')
print('=' * 60)

for name, model in candidates.items():
    scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='f1_macro', n_jobs=-1)
    results[name] = {'mean': scores.mean(), 'std': scores.std()}
    print(f'  {name:20s}  F1={scores.mean():.4f} (±{scores.std():.4f})')

best_name = max(results, key=lambda k: results[k]['mean'])
print(f'\n  🏆 Eng yaxshi: {best_name} (F1={results[best_name]["mean"]:.4f})')

In [ ]:
# Eng yaxshi modelni to'liq o'qitish
best_model = candidates[best_name]
best_model.fit(X_tr, y_tr)
y_pred = best_model.predict(X_te)

acc = accuracy_score(y_te, y_pred)
f1  = f1_score(y_te, y_pred, average='macro')

print(f'Test Accuracy: {acc:.4f}')
print(f'Test F1-macro: {f1:.4f}')
print('\n', classification_report(y_te, y_pred, target_names=le_target.classes_))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_te, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_, ax=ax)
ax.set_title(f'Onamiz Symptom Model — {best_name}\nAccuracy={acc:.3f} | F1={f1:.3f}',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Bashorat', fontsize=11)
ax.set_ylabel('Haqiqiy', fontsize=11)
plt.tight_layout()
plt.savefig('/content/symptom_model_cm.png', dpi=150)
plt.show()

## 🧠 7-QADAM: SHAP — Model nimaga qarab qaror qiladi?

In [ ]:
import shap

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_te[:200])

plt.figure(figsize=(10, 6))
if isinstance(shap_values, list):
    # Multi-class uchun birinchi sinfni ko'rsatamiz
    shap.summary_plot(shap_values[0], X_te[:200],
                      feature_names=FEATURE_COLS, show=False, plot_size=(10,6))
else:
    shap.summary_plot(shap_values, X_te[:200],
                      feature_names=FEATURE_COLS, show=False, plot_size=(10,6))

plt.title('SHAP — Qaysi simptom ko\'proq ta\'sir qiladi?', fontsize=13)
plt.tight_layout()
plt.savefig('/content/shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP grafigi saqlandi')

## 💾 8-QADAM: Modelni saqlash

In [ ]:
import os
os.makedirs('/content/models', exist_ok=True)

artifact = {
    'model': best_model,
    'scaler': scaler,
    'label_encoder': le_target,
    'trimester_encoder': le_trimester,
    'feature_names': FEATURE_COLS,
    'class_labels': list(le_target.classes_),
    'best_model_name': best_name,
    'test_accuracy': float(acc),
    'test_f1_macro': float(f1),
    'cv_results': results,
    'sources': [
        'WHO ANC Guidelines 2016',
        'ACOG Practice Bulletins 2023',
        'FIGO Pre-eclampsia Guidelines 2022',
        'Edinburgh Postnatal Depression Scale'
    ]
}

joblib.dump(artifact, '/content/models/symptom_risk.joblib')
print('✅ Model saqlandi: /content/models/symptom_risk.joblib')
print(f'   Aniqlik: {acc:.1%}')
print(f'   F1-macro: {f1:.4f}')

# Google Drive'ga ko'chirish
import shutil
os.makedirs('/content/drive/MyDrive/onamiz_models', exist_ok=True)
shutil.copy('/content/models/symptom_risk.joblib',
            '/content/drive/MyDrive/onamiz_models/symptom_risk.joblib')
shutil.copy('/content/symptom_model_cm.png',
            '/content/drive/MyDrive/onamiz_models/symptom_model_cm.png')
shutil.copy('/content/shap_importance.png',
            '/content/drive/MyDrive/onamiz_models/shap_importance.png')
print('✅ Drive ga saqlandi: onamiz_models/')

## 🧪 9-QADAM: Real foydalanuvchi stsenariylarini sinash

In [ ]:
def predict_risk(user_input: dict) -> dict:
    """Foydalanuvchi kiritgan ma'lumotlarga asosida xavf darajasini aniqlash."""
    
    risk_messages = {
        'low':       ('🟢', 'yashil',     'Ko\'rsatkichlar normal. Rejali ko\'rikka boring.'),
        'medium':    ('🟡', 'sariq',      'Ba\'zi belgilar bor. Bu hafta shifokorni ko\'ring.'),
        'high':      ('🔴', 'qizil',      'Diqqatni talab qiluvchi belgilar. BUGUN shifokorga boring.'),
        'emergency': ('🚨', 'favqulodda', 'TEZKOR: Hoziroq shifokorga boring yoki tez yordam chaqiring!')
    }
    
    # Input'ni DataFrame'ga aylantirish
    row = {col: user_input.get(col, 0) for col in FEATURE_COLS}
    X_input = pd.DataFrame([row])
    X_input_scaled = scaler.transform(X_input)
    
    probs = best_model.predict_proba(X_input_scaled)[0]
    pred_idx = np.argmax(probs)
    pred_class = le_target.classes_[pred_idx]
    
    emoji, risk_uz, message = risk_messages[pred_class]
    
    return {
        'risk_level': risk_uz,
        'emoji': emoji,
        'message': message,
        'probabilities': {cls: round(float(p), 3) for cls, p in zip(le_target.classes_, probs)},
        'model': best_name
    }


# --- TEST STSENARIYLAR ---
print('=' * 65)
print('  REAL STSENARIYLAR TESTI')
print('=' * 65)

scenarios = [
    {
        'name': '✅ Sog\'lom homilador (28 yosh, T2)',
        'input': {
            'trimester_enc': 1, 'age': 28, 'gestational_week': 18,
            'vaginal_bleeding': 0, 'one_sided_pain': 0, 'nausea_severity': 1,
            'urinary_burning': 0, 'fever': 0, 'dizziness': 0, 'prev_miscarriage': 0,
            'headache_severity': 0, 'visual_disturbance': 0, 'edema_level': 0,
            'fetal_movement': 0, 'epigastric_pain': 0, 'sudden_weight_gain': 0,
            'fluid_leaking': 0, 'fetal_movement_t3': 0, 'contractions': 0,
            'bleeding_with_pain': 0, 'shortness_of_breath': 0,
            'systolic_bp': 115, 'diastolic_bp': 75, 'heart_rate': 80
        }
    },
    {
        'name': '⚠️ Bosh og\'riq + shish (T2)',
        'input': {
            'trimester_enc': 1, 'age': 32, 'gestational_week': 24,
            'vaginal_bleeding': 0, 'one_sided_pain': 0, 'nausea_severity': 2,
            'urinary_burning': 0, 'fever': 0, 'dizziness': 1, 'prev_miscarriage': 0,
            'headache_severity': 2, 'visual_disturbance': 0, 'edema_level': 2,
            'fetal_movement': 0, 'epigastric_pain': 1, 'sudden_weight_gain': 1,
            'fluid_leaking': 0, 'fetal_movement_t3': 0, 'contractions': 0,
            'bleeding_with_pain': 0, 'shortness_of_breath': 0,
            'systolic_bp': 145, 'diastolic_bp': 95, 'heart_rate': 90
        }
    },
    {
        'name': '🚨 Ko\'z oldi uchishi + kuchli bosh og\'riq (eklampsiya xavfi)',
        'input': {
            'trimester_enc': 2, 'age': 35, 'gestational_week': 34,
            'vaginal_bleeding': 0, 'one_sided_pain': 0, 'nausea_severity': 3,
            'urinary_burning': 0, 'fever': 0, 'dizziness': 2, 'prev_miscarriage': 1,
            'headache_severity': 2, 'visual_disturbance': 1, 'edema_level': 3,
            'fetal_movement': 1, 'epigastric_pain': 2, 'sudden_weight_gain': 1,
            'fluid_leaking': 0, 'fetal_movement_t3': 1, 'contractions': 1,
            'bleeding_with_pain': 0, 'shortness_of_breath': 1,
            'systolic_bp': 165, 'diastolic_bp': 112, 'heart_rate': 102
        }
    },
    {
        'name': '🚨 Qin suvi oqishi (PPROM)',
        'input': {
            'trimester_enc': 1, 'age': 26, 'gestational_week': 22,
            'vaginal_bleeding': 1, 'one_sided_pain': 0, 'nausea_severity': 0,
            'urinary_burning': 0, 'fever': 0, 'dizziness': 0, 'prev_miscarriage': 0,
            'headache_severity': 0, 'visual_disturbance': 0, 'edema_level': 1,
            'fetal_movement': 0, 'epigastric_pain': 0, 'sudden_weight_gain': 0,
            'fluid_leaking': 1, 'fetal_movement_t3': 0, 'contractions': 0,
            'bleeding_with_pain': 0, 'shortness_of_breath': 0,
            'systolic_bp': 118, 'diastolic_bp': 76, 'heart_rate': 84
        }
    }
]

for s in scenarios:
    result = predict_risk(s['input'])
    print(f"\n📋 {s['name']}")
    print(f"   {result['emoji']} Natija: {result['risk_level'].upper()}")
    print(f"   💬 {result['message']}")
    print(f"   📊 Ehtimollar: {result['probabilities']}")

## 📤 10-QADAM: Kaggle dataseti bilan birlashtirish (Bonus)

In [ ]:
# Agar Kaggle Maternal Risk dataseti mavjud bo'lsa, birlashtirish
KAGGLE_PATH = '/content/drive/MyDrive/onamiz_data/Maternal Health Risk Data Set.csv'

if os.path.exists(KAGGLE_PATH):
    kaggle_df = pd.read_csv(KAGGLE_PATH)
    print('Kaggle dataset:', kaggle_df.shape)
    print(kaggle_df['RiskLevel'].value_counts())
    
    # Kaggle ma'lumotlarini bizning formatga o'tkazish
    kaggle_mapped = pd.DataFrame({
        'trimester_enc': 1,  # T2 deb faraz qilamiz
        'age': kaggle_df['Age'],
        'gestational_week': 20,
        'systolic_bp': kaggle_df['SystolicBP'],
        'diastolic_bp': kaggle_df['DiastolicBP'],
        'heart_rate': kaggle_df['HeartRate'],
        'fever': (kaggle_df['BodyTemp'] > 99.5).astype(int),
        # Qolgan simptomlar nol (ular bu ma'lumotni yig'magan)
        **{col: 0 for col in FEATURE_COLS if col not in
           ['trimester_enc','age','gestational_week','systolic_bp','diastolic_bp','heart_rate','fever']}
    })
    
    risk_map = {'low risk': 'low', 'mid risk': 'medium', 'high risk': 'high'}
    kaggle_mapped['risk_level'] = kaggle_df['RiskLevel'].map(risk_map)
    
    print('\nKaggle dataseti bizning formatga moslashtirildi:', kaggle_mapped.shape)
else:
    print('Kaggle dataseti topilmadi. Faqat sintetik dataset ishlatiladi.')
    print('Drive ga yuklash uchun: /content/drive/MyDrive/onamiz_data/ papkasiga joylashtiring')

## ✅ Xulosa

| | Natija |
|---|---|
| **Model** | XGBoost / LightGBM |
| **Features** | 24 ta (simptomlar + vital belgilar) |
| **Sinflar** | 4 ta (low/medium/high/emergency) |
| **Manba** | WHO ANC 2016 + ACOG + FIGO 2022 + EPDS |
| **Chiqish** | Doktorga borish kerakmi: ha/yo'q + rang |

### Keyingi qadamlar:
1. Real foydalanuvchi ma'lumotlari to'plangach modelni yangilash
2. O'zbek ginekolog bilan savollarni tekshirish
3. FastAPI ga integratsiya qilish
4. Flutter ilovaga ulash